In [1]:
import re
import pandas as pd
from datetime import datetime
from telegram import Update
from telegram.ext import ApplicationBuilder, ContextTypes, MessageHandler, CommandHandler, filters

# Datenstruktur zum Speichern der Beträge
data = []

In [2]:
async def message_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    global data
    message = update.message.text
    user = update.message.from_user.username or update.message.from_user.first_name

    # Debug: Eingehende Nachricht anzeigen
    print(f"Empfangene Nachricht: {message}")

    # Suche nach Geldbeträgen mit verbessertem Regex
    pattern = r"(\d+(?:[\.,]\d{1,2})?)\s?(€|EUR|USD|CHF)?"
    match = re.search(pattern, message)

    if match:
        # Debug: Gefundene Gruppen anzeigen
        print(f"Gefundener Betrag: {match.group(1)}")
        print(f"Gefundene Währung: {match.group(2)}")

        # Betrag und Währung extrahieren
        amount = float(match.group(1).replace(",", "."))
        currency = match.group(2) or "EUR"

        # Daten speichern
        data.append({
            "user": user,
            "amount": amount,
            "currency": currency,
            "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        # Debug: Aktuelle Daten anzeigen
        print(f"Aktuelle Daten: {data}")

        await update.message.reply_text(f"{user}, ich habe {amount} {currency} gespeichert!")
    else:
        # Debug: Regex hat nichts gefunden
        print(f"Regex hat keinen Betrag gefunden. Nachricht: {message}")
        await update.message.reply_text("Kein gültiger Betrag erkannt.")

In [3]:
async def summary_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    global data

    if not data:
        await update.message.reply_text("Keine Daten vorhanden.")
        return

    # Erstelle einen DataFrame aus den Daten
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.to_period('M')

    # Summiere die Beträge
    summary = df.groupby(['user', 'month', 'currency'])['amount'].sum().reset_index()

    # Ausgabe formatieren
    summary_text = "\\nZusammenfassung pro Person und Monat:\\n"
    for _, row in summary.iterrows():
        summary_text += f"{row['user']} - {row['month']} - {row['amount']:.2f} {row['currency']}\\n"
    print(data)
    await update.message.reply_text(summary_text)
   

In [4]:

import nest_asyncio
import asyncio
from telegram import Bot

# nest_asyncio aktivieren
nest_asyncio.apply()

# Bot-Token
TOKEN = "7609808796:AAH1BgMYmjHRbDbn5ouhnrGcSuvuYajOr5I"
bot = Bot(token=TOKEN)

# Chat-ID der Gruppe
CHAT_ID = "2449977748"

# Abrufen der Nachrichten aus der Chathistory
async def fetch_chat_history():
    updates = await bot.get_updates()  # Abrufen neuer Nachrichten
    for update in updates:
        # Überprüfe, ob die Nachricht aus der richtigen Gruppe stammt
        if update.message and update.message.chat.id == CHAT_ID:
            print(f"Nachricht: {update.message.text} von {update.message.from_user.username}")

# Event-Loop verwenden
await fetch_chat_history()

# Executing the bot

In [6]:
from telegram.ext import ApplicationBuilder, Application

TOKEN = "7609808796:AAH1BgMYmjHRbDbn5ouhnrGcSuvuYajOr5I"  # Ersetze mit deinem Bot-Token

# Bot-Anwendung erstellen
app = ApplicationBuilder().token(TOKEN).build()

# Handler hinzufügen
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, message_handler))
app.add_handler(CommandHandler("summary", summary_handler))

# Bot in Jupyter kompatibel starten
import nest_asyncio
import asyncio

# Nest AsyncIO aktivieren (nur in Jupyter notwendig)
nest_asyncio.apply()

# Starten
print("Bot läuft...")
await app.run_polling()

Bot läuft...
Empfangene Nachricht: 30€
Gefundener Betrag: 30
Gefundene Währung: €
Aktuelle Daten: [{'user': 'Moses', 'amount': 30.0, 'currency': '€', 'date': '2025-01-25 21:04:24'}]
Empfangene Nachricht: 20.e
Gefundener Betrag: 20
Gefundene Währung: None
Aktuelle Daten: [{'user': 'Moses', 'amount': 30.0, 'currency': '€', 'date': '2025-01-25 21:04:24'}, {'user': 'Moses', 'amount': 20.0, 'currency': 'EUR', 'date': '2025-01-25 21:04:29'}]
[{'user': 'Moses', 'amount': 30.0, 'currency': '€', 'date': '2025-01-25 21:04:24'}, {'user': 'Moses', 'amount': 20.0, 'currency': 'EUR', 'date': '2025-01-25 21:04:29'}]


RuntimeError: Cannot close a running event loop